In [142]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import gradio as gr
openai = OpenAI()

In [143]:
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'


In [144]:
system_message = "You are a code converter assistant. This is your only job. Also be a little sassy!"

In [145]:
import json

def save_context_to_file(context, file_name="context.json"):
    with open(file_name, "w") as file:
        json.dump(context, file)


def load_context(file_name="context.json"):
    try:
        if os.path.exists(file_name) and os.path.getsize(file_name) > 0:
            with open(file_name, "r") as file:
                return json.load(file)
        else:
            return []
    except (json.JSONDecodeError, ValueError):
        return []
    

def trim_context(context, max_tokens=4096):
        while len(json.dumps(context)) > max_tokens:
            context.pop(0)  # Remove the oldest message
        return context

In [146]:
def summarize_conversation(messages, model="gpt-3.5-turbo"):
    
    full_conversation = "\n".join([f"{m['role']}: {m['content']}" for m in messages])
    summary = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Summarize the following conversation into key points:"},
            {"role": "user", "content": full_conversation}
        ]
    )
    
    
    return summary.choices[0].message.content

In [147]:
def chat(message, history):
    
    # max_context_tokens = 1500
    
    context = load_context()
    
    if not context:
        context.append({"role": "system", "content": system_message})
    
    for user_message, assistant_message in history:
        context.append({"role": "user", "content": user_message})
        context.append({"role": "assistant", "content": assistant_message})
    
    
    context.append({"role": "user", "content": message})
    
    # if len(json.dumps(context)) > max_context_tokens:
    
    summarized_content = summarize_conversation(context)
    summerized_points = summarized_content.split("\n")    
    context = [{"role": "system", "content": system_message}]
    for point in summerized_points:
        context.append({"role": "assistant", "content": point})


    stream = openai.chat.completions.create(model=MODEL, messages=context, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response
    
    context.append({"role": "assistant", "content": response})
    
    save_context_to_file(context)

In [148]:
gr.ChatInterface(fn=chat).launch()


/Users/adamlindberg/Documents/VSCode/code-conversion-bot/env/lib/python3.13/site-packages/gradio/components/chatbot.py:279: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7885

To create a public link, set `share=True` in `launch()`.
